# SciBERT Edge Scoring v2 - Masked Label

Notebook này tải P5 transitive-pruned output từ Google Drive, dùng `allenai/scibert_scivocab_uncased` dạng masked language model để chấm prerequisite edge.

Bản này hỏi model trực tiếp: với source/target concept, claim prerequisite là `[MASK]`, rồi so xác suất nhãn `valid` vs `invalid`. Đồng thời chấm cả chiều đảo để tạo `bidirectional_score`.

In [1]:
DRIVE_URL = "https://drive.google.com/file/d/1HyWTp9bJv2lg1EF17H_vxVSZnYLY4lDm/view?usp=sharing"
MODEL_ID = "allenai/scibert_scivocab_uncased"
MAX_LENGTH = 512
BATCH_SIZE = 16
OUTPUT_PATH = "/content/scibert_edge_scores_masked_v2.json"

In [2]:
!pip -q install -U "transformers>=4.57.0" accelerate gdown safetensors

In [3]:
import json
import math
import re
import shutil
import zipfile
from pathlib import Path

import requests

download_path = Path("/content/p5_drive_input")
extract_dir = Path("/content/p5_drive_extract")
if download_path.exists():
    download_path.unlink()
if extract_dir.exists():
    shutil.rmtree(extract_dir)

def extract_drive_file_id(url: str) -> str:
    patterns = [r"/file/d/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Cannot extract Google Drive file id from: {url}")

def download_from_drive(file_id: str, output_path: Path) -> None:
    import gdown

    result = gdown.download(id=file_id, output=str(output_path), quiet=False)
    if result and output_path.exists() and output_path.stat().st_size > 0:
        return

    session = requests.Session()
    url = "https://drive.google.com/uc?export=download"
    response = session.get(url, params={"id": file_id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break
    if token:
        response = session.get(url, params={"id": file_id, "confirm": token}, stream=True)
    response.raise_for_status()
    with output_path.open("wb") as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError("Google Drive download produced an empty file. Check sharing permissions or upload manually.")

drive_file_id = extract_drive_file_id(DRIVE_URL)
print("Drive file id:", drive_file_id)
download_from_drive(drive_file_id, download_path)
print("Downloaded:", download_path, "bytes=", download_path.stat().st_size)

if zipfile.is_zipfile(download_path):
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(download_path) as zf:
        zf.extractall(extract_dir)
    json_files = sorted(extract_dir.rglob("*.json"))
    if not json_files:
        raise FileNotFoundError("Zip downloaded from Drive has no .json files")
    input_json_path = json_files[0]
else:
    input_json_path = download_path

payload = json.loads(input_json_path.read_text(encoding="utf-8"))
print("Input JSON:", input_json_path)
print("Top-level keys:", list(payload.keys()))

Drive file id: 1HyWTp9bJv2lg1EF17H_vxVSZnYLY4lDm


Downloading...
From: https://drive.google.com/uc?id=1HyWTp9bJv2lg1EF17H_vxVSZnYLY4lDm
To: /content/p5_drive_input
100%|██████████| 169k/169k [00:00<00:00, 3.78MB/s]

Downloaded: /content/p5_drive_input bytes= 169261
Input JSON: /content/p5_drive_input
Top-level keys: ['run_id', 'stage_id', 'clean_candidate_edges', 'pruned_edges', 'adjudication_trace', 'transitive_prune_summary']


In [ ]:
clean_edges = payload.get("clean_candidate_edges")
if not isinstance(clean_edges, list):
    raise ValueError("Expected top-level clean_candidate_edges[] in P5 transitive-pruned output")

global_kps = payload.get("global_kps") or payload.get("concepts_kp_global") or []
kp_index = {}
for kp in global_kps:
    if isinstance(kp, dict):
        kp_id = kp.get("global_kp_id") or kp.get("kp_id")
        if kp_id:
            kp_index[kp_id] = kp

def readable_kp_id(kp_id: str) -> str:
    return re.sub(r"[_-]+", " ", kp_id.removeprefix("kp_")).strip()

def kp_text(kp_id: str) -> str:
    kp = kp_index.get(kp_id, {})
    name = kp.get("name") or kp.get("canonical_name") or readable_kp_id(kp_id)
    desc = kp.get("description") or kp.get("global_description") or ""
    role = kp.get("structural_role") or ""
    importance = kp.get("importance_level") or ""
    parts = [f"Concept: {name}."]
    if desc:
        parts.append(f"Description: {desc}")
    if role or importance:
        parts.append(f"Role: {role}. Importance: {importance}.")
    return " ".join(parts)

def claim_prompt(edge: dict, *, reverse: bool = False) -> str:
    source = edge["target_kp_id"] if reverse else edge["source_kp_id"]
    target = edge["source_kp_id"] if reverse else edge["target_kp_id"]
    rationale = edge.get("keep_rationale", "")
    return (
        "A prerequisite relation is valid when a learner should understand the source concept before the target concept. "
        "It is invalid when the concepts merely co-occur, are reversed, are the same concept, or only share historical context.\n\n"
        f"Source concept:\n{kp_text(source)}\n\n"
        f"Target concept:\n{kp_text(target)}\n\n"
        f"Proposed rationale:\n{rationale}\n\n"
        "The proposed prerequisite relation is [MASK]."
    )

print("Clean edges:", len(clean_edges))
print("KP catalog entries available:", len(kp_index))
print("Sample prompt:\n", claim_prompt(clean_edges[0])[:1200])

Clean edges: 93
KP catalog entries available: 0
Sample prompt:
 A prerequisite relation is valid when a learner should understand the source concept before the target concept. It is invalid when the concepts merely co-occur, are reversed, are the same concept, or only share historical context.

Source concept:
Concept: 3d datasets and ai geometry task space.

Target concept:
Concept: multi view and voxel based 3d deep learning.

Proposed rationale:
3D datasets and AI+geometry task space provides prerequisite conceptual machinery, motivation, or implementation context needed to understand Multi-view and voxel-based 3D deep learning; the target specializes, applies, or operationalizes the source rather than merely co-occurring with it.

The proposed prerequisite relation is [MASK].


In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device=", device, "dtype=", dtype)
if device == "cuda":
    print("gpu=", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    attn_implementation="sdpa",
).to(device)
model.eval()

def single_token_id(label: str) -> int | None:
    ids = tokenizer.encode(label, add_special_tokens=False)
    return ids[0] if len(ids) == 1 else None

label_candidates = [("valid", "invalid"), ("true", "false"), ("yes", "no")]
label_pair = None
for pos, neg in label_candidates:
    pos_id = single_token_id(pos)
    neg_id = single_token_id(neg)
    print("label candidate", pos, pos_id, neg, neg_id)
    if pos_id is not None and neg_id is not None:
        label_pair = (pos, neg, pos_id, neg_id)
        break
if label_pair is None:
    raise RuntimeError("No single-token positive/negative label pair found for this tokenizer")

POS_LABEL, NEG_LABEL, POS_ID, NEG_ID = label_pair
print("Using labels:", POS_LABEL, POS_ID, NEG_LABEL, NEG_ID)

def score_prompts(prompts, batch_size=BATCH_SIZE):
    results = []
    for start in range(0, len(prompts), batch_size):
        batch = prompts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)
        mask_positions = (encoded["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=False)
        if mask_positions.shape[0] != len(batch):
            raise RuntimeError(f"Expected one mask per prompt, found {mask_positions.shape[0]} masks for batch size {len(batch)}")
        with torch.no_grad():
            logits = model(**encoded).logits
        for row, pos in mask_positions:
            two_logits = torch.stack([logits[row, pos, POS_ID], logits[row, pos, NEG_ID]]).float()
            probs = torch.softmax(two_logits, dim=0)
            score = probs[0].item()
            results.append({
                "score": score,
                "positive_logit": two_logits[0].item(),
                "negative_logit": two_logits[1].item(),
                "positive_prob": probs[0].item(),
                "negative_prob": probs[1].item(),
            })
    return results

print("Model loaded")

device= cuda dtype= torch.float16
gpu= Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

BertForMaskedLM LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


label candidate valid 2119 invalid 17933
Using labels: valid 2119 invalid 17933
Model loaded


In [6]:
forward_prompts = [claim_prompt(edge, reverse=False) for edge in clean_edges]
reverse_prompts = [claim_prompt(edge, reverse=True) for edge in clean_edges]

forward_scores = score_prompts(forward_prompts)
reverse_scores = score_prompts(reverse_prompts)

def review_status(edge_strength: float, bidirectional_score: float, keep_confidence: str) -> str:
    if edge_strength >= 0.5 and bidirectional_score <= 0.3:
        return "not_required"
    if edge_strength >= 0.3:
        return "optional"
    if keep_confidence == "low":
        return "reject_candidate"
    return "deferred"

scored_edges = []
for edge, fs, rs in zip(clean_edges, forward_scores, reverse_scores):
    edge_strength = fs["score"]
    bidirectional_score = rs["score"]
    direction_margin = edge_strength - bidirectional_score
    scored = dict(edge)
    scored.update({
        "edge_strength": round(edge_strength, 4),
        "bidirectional_score": round(bidirectional_score, 4),
        "direction_margin": round(direction_margin, 4),
        "scibert_model": MODEL_ID,
        "scibert_scoring_method": "masked_label_valid_invalid_v2",
        "scibert_positive_label": POS_LABEL,
        "scibert_negative_label": NEG_LABEL,
        "scibert_review_status": review_status(edge_strength, bidirectional_score, edge.get("keep_confidence", "medium")),
        "scibert_debug": {
            "forward_positive_logit": round(fs["positive_logit"], 4),
            "forward_negative_logit": round(fs["negative_logit"], 4),
            "reverse_positive_logit": round(rs["positive_logit"], 4),
            "reverse_negative_logit": round(rs["negative_logit"], 4),
        },
    })
    scored_edges.append(scored)

print("Scored edges:", len(scored_edges))
print(json.dumps(scored_edges[0], indent=2)[:1600])

Scored edges: 93
{
  "source_kp_id": "kp_3d_datasets_and_ai_geometry_task_space",
  "target_kp_id": "kp_multi_view_and_voxel_based_3d_deep_learning",
  "edge_scope": "intra_course",
  "provenance": "llm_cross_check",
  "keep_confidence": "medium",
  "keep_rationale": "3D datasets and AI+geometry task space provides prerequisite conceptual machinery, motivation, or implementation context needed to understand Multi-view and voxel-based 3D deep learning; the target specializes, applies, or operationalizes the source rather than merely co-occurring with it.",
  "expected_directionality": "moderate",
  "review_status": "optional",
  "ready_for_modernbert": true,
  "edge_strength": 0.8606,
  "bidirectional_score": 0.8459,
  "direction_margin": 0.0147,
  "scibert_model": "allenai/scibert_scivocab_uncased",
  "scibert_scoring_method": "masked_label_valid_invalid_v2",
  "scibert_positive_label": "valid",
  "scibert_negative_label": "invalid",
  "scibert_review_status": "optional",
  "scibert_de

In [7]:
from collections import Counter

strengths = [e["edge_strength"] for e in scored_edges]
bidir = [e["bidirectional_score"] for e in scored_edges]
margins = [e["direction_margin"] for e in scored_edges]

output = {
    "run_id": payload.get("run_id", "p5_cs224n_cs231n"),
    "stage_id": "scibert_edge_scoring",
    "source_input_path": str(input_json_path),
    "model_id": MODEL_ID,
    "max_length": MAX_LENGTH,
    "scoring_method": "masked_label_valid_invalid_v2",
    "positive_label": POS_LABEL,
    "negative_label": NEG_LABEL,
    "scored_edges": scored_edges,
    "score_summary": {
        "edge_count": len(scored_edges),
        "scibert_review_status_distribution": dict(Counter(e["scibert_review_status"] for e in scored_edges)),
        "edge_strength_min": round(min(strengths), 4),
        "edge_strength_max": round(max(strengths), 4),
        "edge_strength_mean": round(sum(strengths) / len(strengths), 4),
        "bidirectional_score_min": round(min(bidir), 4),
        "bidirectional_score_max": round(max(bidir), 4),
        "bidirectional_score_mean": round(sum(bidir) / len(bidir), 4),
        "direction_margin_min": round(min(margins), 4),
        "direction_margin_max": round(max(margins), 4),
        "direction_margin_mean": round(sum(margins) / len(margins), 4),
    },
}

Path(OUTPUT_PATH).write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(output["score_summary"], indent=2))
print("Wrote", OUTPUT_PATH)

{
  "edge_count": 93,
  "scibert_review_status_distribution": {
    "optional": 93
  },
  "edge_strength_min": 0.4922,
  "edge_strength_max": 0.8606,
  "edge_strength_mean": 0.7706,
  "bidirectional_score_min": 0.5254,
  "bidirectional_score_max": 0.8587,
  "bidirectional_score_mean": 0.7668,
  "direction_margin_min": -0.0332,
  "direction_margin_max": 0.0451,
  "direction_margin_mean": 0.0038
}
Wrote /content/scibert_edge_scores_masked_v2.json


In [8]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_output = Path("/content/drive/MyDrive/scibert_edge_scores_masked_v2.json")
    shutil.copyfile(OUTPUT_PATH, drive_output)
    print("Saved to Google Drive:", drive_output)
except Exception as exc:
    print("Drive save skipped:", exc)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive: /content/drive/MyDrive/scibert_edge_scores_masked_v2.json
